# Notebook 07 — Comparativo Final: Baseline vs. Modelos com Sentimento (H=1)

> **Pré-requisito:** executar o Notebook 02 e o Notebook 06 até as células de exportação, gerando os arquivos:
> - `resultados_baseline_h1.csv`
> - `resultados_sentimento_h1.csv`

## Regras metodológicas
- Nenhum modelo é reestimado aqui.
- O critério de comparação principal é o **RMSE** no conjunto de teste (com correção de bias).
- Para os modelos com sentimento, são exibidos os **top 3 por família** (menor RMSE com bias).

## 1. Importações

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.linear_model import LinearRegression

COLOR = "#0e5764"  # padrão visual do Notebook 02

## 2. Leitura dos resultados exportados

In [ ]:
# ── Baseline (Notebook 02) ───────────────────────────────────────────────────
df_base = pd.read_csv("resultados_baseline_h1.csv")

# Padroniza nome da coluna de modelo (pode vir como "Modelo" ou "index")
if "Modelo" not in df_base.columns and "index" in df_base.columns:
    df_base = df_base.rename(columns={"index": "Modelo"})

# Identifica família a partir do nome do modelo
def _familia(nome):
    n = str(nome).upper()
    if "ARIMA" in n: return "ARIMA"
    if "SVR"   in n: return "SVR"
    if "XGBOOST" in n or "XGB" in n: return "XGBoost"
    return "Outro"

df_base["Família"]  = df_base["Modelo"].apply(_familia)
df_base["Com bias"] = df_base["Modelo"].str.contains("bias", case=False, na=False)
df_base["Com bias"] = df_base["Com bias"].map({True: "Sim", False: "Não"})

print(f"Baseline carregado: {df_base.shape[0]} linhas")
display(df_base.round(6))

# ── Sentimento (Notebook 06) ─────────────────────────────────────────────────
df_sent = pd.read_csv("resultados_sentimento_h1.csv")

print(f"\nSentimento carregado: {df_sent.shape[0]} linhas")
display(df_sent.round(6))

## 3. Tabela comparativa final

In [ ]:
# ── Baseline com bias ────────────────────────────────────────────────────────
base_cb = (
    df_base[df_base["Com bias"] == "Sim"]
    .sort_values("RMSE")
    .reset_index(drop=True)
    .copy()
)

# ── Sentimento com bias ──────────────────────────────────────────────────────
sent_cb = (
    df_sent[df_sent["Com bias"] == "Sim"]
    .sort_values(["Família", "RMSE"])
    .reset_index(drop=True)
    .copy()
)
sent_cb["Rank"] = sent_cb.groupby("Família").cumcount() + 1

# ── ΔRMSE em relação ao baseline da mesma família ────────────────────────────
rmse_baseline = base_cb.set_index("Família")["RMSE"].to_dict()

sent_cb["ΔRMSE vs Baseline"] = sent_cb.apply(
    lambda r: round(r["RMSE"] - rmse_baseline.get(r["Família"], np.nan), 6), axis=1
)
sent_cb["Melhorou?"] = sent_cb["ΔRMSE vs Baseline"].apply(
    lambda v: "✅ Sim" if v < 0 else "❌ Não"
)

# ── Monta tabela unificada ───────────────────────────────────────────────────
linhas_base = base_cb[["Família","Modelo","RMSE","MAE","R2",
                         "Bias (obs - prev)","Slope pred~obs"]].copy()
linhas_base.insert(1, "Tipo", "Baseline")
linhas_base.insert(3, "Tipo Relatório",    "—")
linhas_base.insert(4, "Modelo Sentimento", "—")
linhas_base["Rank"]                = "—"
linhas_base["ΔRMSE vs Baseline"]   = np.nan
linhas_base["Melhorou?"]           = "—"

linhas_sent = sent_cb[["Família","Rank","Tipo Relatório","Modelo Sentimento",
                         "Configuração","RMSE","MAE","R2",
                         "Bias (obs - prev)","Slope pred~obs",
                         "ΔRMSE vs Baseline","Melhorou?"]].copy()
linhas_sent.insert(1, "Tipo",  linhas_sent["Rank"].apply(lambda r: f"+ Sentimento #{r}"))
linhas_sent.insert(2, "Modelo", linhas_sent.apply(
    lambda r: f"{r['Família']} + {r['Tipo Relatório']} + {r['Modelo Sentimento']}", axis=1))

COLS_COMP = ["Família","Tipo","Modelo Sentimento","Tipo Relatório",
             "RMSE","MAE","R2","Bias (obs - prev)","Slope pred~obs",
             "ΔRMSE vs Baseline","Melhorou?"]

df_comp = (
    pd.concat([linhas_base[COLS_COMP], linhas_sent[COLS_COMP]], ignore_index=True)
    .sort_values(["Família","RMSE"])
    .reset_index(drop=True)
)

print("=== Comparativo Final — Baseline vs. Sentimento (H=1, com bias) ===")
display(df_comp.round(6))

## 4. Gráfico comparativo de RMSE

In [ ]:
familias = ["ARIMA", "SVR", "XGBoost"]
ALPHA_SENT = [0.78, 0.56, 0.38]

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)
fig.suptitle("RMSE — Baseline vs. Modelos com Sentimento (H=1, com bias)", fontsize=13, y=1.02)

for ax, familia in zip(axes, familias):
    rmse_base = rmse_baseline.get(familia, np.nan)
    top3_f = sent_cb[sent_cb["Família"] == familia].sort_values("RMSE").reset_index(drop=True)

    labels = [f"Baseline"]
    rmses  = [rmse_base]
    for _, row in top3_f.iterrows():
        labels.append(f"#{int(row['Rank'])} {row['Modelo Sentimento']}\n{str(row['Tipo Relatório'])[:10]}")
        rmses.append(row["RMSE"])

    n = len(rmses)
    x = np.arange(n)
    alphas = [1.0] + ALPHA_SENT[:n-1]

    bars = ax.bar(x, rmses, color=COLOR, width=0.55, zorder=3)
    for bar, a in zip(bars, alphas):
        bar.set_alpha(a)

    # valor acima de cada barra
    for bar, v in zip(bars, rmses):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.0012,
                f"{v:.4f}", ha="center", va="bottom", fontsize=8, color="#333333")

    # linha de referência baseline
    ax.axhline(rmse_base, color=COLOR, linestyle="--", lw=1.2, alpha=0.5, zorder=2)

    # delta no rodapé das barras de sentimento
    for bar, v in zip(bars[1:], rmses[1:]):
        delta = v - rmse_base
        cor   = "#1a7a3c" if delta < 0 else "#b03030"
        sinal = "▼" if delta < 0 else "▲"
        ax.text(bar.get_x() + bar.get_width()/2, 0.001,
                f"{sinal}{abs(delta):.4f}", ha="center", va="bottom",
                fontsize=7.5, color=cor, zorder=4)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_title(familia, fontsize=11, fontweight="bold")
    ax.set_ylabel("RMSE" if familia == "ARIMA" else "")
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
    ax.grid(axis="y", alpha=0.25, zorder=1)
    ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.savefig("rmse_comparativo_nb07.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Gráficos previsto vs. observado — com e sem bias

In [ ]:
# ── Lê as predições do CSV de sentimento (se disponíveis) ────────────────────
# Para gerar o scatter é necessário ter as colunas de previsão salvas.
# Como o CSV exportado não contém as séries y_true/y_pred, este gráfico
# usa apenas os dados do CSV de baseline que já traz as métricas resumidas.
# Para o scatter completo, execute o Notebook 06 e copie as predições manualmente.

print("ℹ️  O gráfico previsto vs. observado requer as séries de previsão (y_true / y_pred).")
print("   Execute o Notebook 06 para obtê-las. As métricas comparativas estão na tabela acima.")

## 6. Conclusões

In [ ]:
# ── 1. O sentimento melhorou por família? ────────────────────────────────────
print("=" * 68)
print("CONCLUSÃO 1 — O sentimento melhorou a performance dos modelos?")
print("=" * 68)

melhora = []
for familia in familias:
    rmse_b = rmse_baseline.get(familia, np.nan)
    top1   = sent_cb[sent_cb["Família"]==familia].sort_values("RMSE").iloc[0]
    delta  = top1["RMSE"] - rmse_b
    ok     = delta < 0
    melhora.append(ok)
    tag    = "▼ MELHOROU" if ok else "▲ PIOROU"
    print(f"  {familia:8s} | Baseline={rmse_b:.5f}  Melhor Sent.={top1['RMSE']:.5f}"
          f"  ΔRMSE={delta:+.5f}  {tag}")

print()
if all(melhora):
    print("  ✅ O sentimento melhorou a performance em TODAS as famílias.")
elif any(melhora):
    print(f"  ⚠️  O sentimento melhorou em {sum(melhora)}/3 famílias.")
else:
    print("  ❌ O sentimento NÃO melhorou a performance em nenhuma família.")

# ── 2. Qual sentimento foi mais eficaz? ───────────────────────────────────────
print()
print("=" * 68)
print("CONCLUSÃO 2 — Qual modelo de sentimento forneceu o melhor resultado?")
print("=" * 68)
for familia in familias:
    top1 = sent_cb[sent_cb["Família"]==familia].sort_values("RMSE").iloc[0]
    print(f"  {familia:8s} | {top1['Modelo Sentimento']:22s} ({top1['Tipo Relatório']})  RMSE={top1['RMSE']:.5f}")

melhor = sent_cb.sort_values("RMSE").iloc[0]
print(f"\n  🏆 Melhor global: {melhor['Modelo Sentimento']} ({melhor['Tipo Relatório']}) "
      f"via {melhor['Família']} — RMSE={melhor['RMSE']:.5f}")

# ── 3. Sentimento é globalmente melhor? ───────────────────────────────────────
print()
print("=" * 68)
print("CONCLUSÃO 3 — O sentimento fornece um resultado melhor (visão geral)?")
print("=" * 68)
rmse_min_b = min(rmse_baseline.values())
rmse_min_s = sent_cb["RMSE"].min()
delta_g    = rmse_min_s - rmse_min_b
print(f"  RMSE mínimo baseline:     {rmse_min_b:.5f}")
print(f"  RMSE mínimo c/sentimento: {rmse_min_s:.5f}")
print(f"  Δ global: {delta_g:+.5f}")
if delta_g < 0:
    print(f"\n  ✅ O sentimento é MELHOR — redução de {abs(delta_g)/rmse_min_b*100:.1f}% no RMSE mínimo.")
else:
    print(f"\n  ❌ O sentimento NÃO melhora o resultado global no conjunto de teste analisado.")

In [ ]:
# ── Tabela-resumo ─────────────────────────────────────────────────────────────
resumo = []
for familia in familias:
    rmse_b = rmse_baseline.get(familia, np.nan)
    top1   = sent_cb[sent_cb["Família"]==familia].sort_values("RMSE").iloc[0]
    delta  = top1["RMSE"] - rmse_b
    resumo.append({
        "Família":               familia,
        "RMSE Baseline":         round(rmse_b, 6),
        "RMSE Melhor Sentimento":round(top1["RMSE"], 6),
        "ΔRMSE":                 round(delta, 6),
        "Melhorou?":             "✅ Sim" if delta < 0 else "❌ Não",
        "Melhor Modelo Sent.":   top1["Modelo Sentimento"],
        "Tipo Relatório":        top1["Tipo Relatório"],
    })

print("=== Resumo das Conclusões ===")
display(pd.DataFrame(resumo))

## 7. Leitura metodológica para o artigo

Este notebook reúne os resultados finais dos modelos autorregressivos (Notebook 02) e dos modelos com variáveis de sentimento (Notebook 06), comparando-os pelo **RMSE fora da amostra** com correção de viés estimada exclusivamente no treino.

Para cada família de modelo — ARIMA/ARIMAX, SVR e XGBoost —, o baseline corresponde à especificação selecionada no Notebook 02. Os modelos com sentimento herdam essas configurações sem reestimação, acrescentando apenas a janela de sentimento paralela à janela da inadimplência. A análise de ΔRMSE indica se o acréscimo de informação textual foi capaz de reduzir o erro preditivo em relação ao baseline puramente autorregressivo.